# Figure 5: sensitivity to the MM smoothing parameter

Run seven methods at 140 logarithmically spaced $\epsilon$ values from $10^{-12}$ to
$10^2$, with 150 iterations per solve. Flexible methods can stop earlier at their
breakdown tolerance. The outputs are `results/figure_05.csv` and `results/figure_05.png`.

This is the longest 1D notebook. Progress is printed after each epsilon value;
full solver messages go to `results/figure_05_solver.log`.


In [ ]:
print("Device: CPU")

from pathlib import Path
import sys
import os

# Configure libraries loaded later as well as those already loaded by the kernel.
for _name in ["OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_name] = "8"

# Launch Jupyter from the repository root.
ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import contextlib
import numpy as np
import pandas as pd
import scipy.sparse as sps
from scipy.fft import dct, idct
import matplotlib.pyplot as plt
import operators
from threadpoolctl import threadpool_limits

# Use eight BLAS threads, set before constructing data.
_blas_threads = threadpool_limits(limits=8)

# Figure color palette.
colors = [(106,192,205), (235,186,63), (172,46,55), (32,82,145),
          (83,165,106), (239,145,75), (208,92,89), (80,67,157),
          (82,82,82), (53,53,53), (30,27,28)]
plt.rc("axes", prop_cycle=plt.cycler(color=[tuple(v/255 for v in c) for c in colors]))


## Synthetic data and first-difference prior

Every epsilon value uses the same seed-0 data realization at 3% relative noise.
The first-difference inverse is stored as a dense matrix.


In [ ]:
# Synthetic piecewise-constant signal and the first 50 orthonormal DCT coefficients.
n = 1000
ground_truth = np.empty(n)
ground_truth[:250] = 2.5
ground_truth[250:400] = -0.5
ground_truth[400:625] = 4.5
ground_truth[625:825] = 2.5
ground_truth[825:] = 0.5

class DCT1D(operators._CustomLinearOperator):
    def __init__(self, n):
        super().__init__(
            (n, n),
            lambda x: dct(np.squeeze(x), type=2, norm="ortho"),
            lambda x: idct(np.squeeze(x), type=2, norm="ortho"),
            device="cpu", dtype=None,
        )

mask = np.zeros(n)
mask[:50] = 1
A = operators.Subsampling1DOperator(mask) @ DCT1D(n)
b_true = A @ ground_truth
# RandomState uses the same random-number sequence as np.random.seed(0).
noise = np.random.RandomState(0).normal(size=A.shape[0])
noise /= np.linalg.norm(noise)
noise *= 0.03 * np.linalg.norm(b_true)
b = b_true + noise
noise_var = (0.03 * np.linalg.norm(b_true))**2 / A.shape[0]

Lmat, _ = operators.first_order_derivative_1d(n, boundary="zero")
L = operators.MatrixLinearOperator(Lmat)
# Use a dense inverse with a fixed BLAS evaluation order.
# An algebraically equivalent cumulative-sum inverse changes tiny roundoff-sensitive curves.
Lpinv = operators.MatrixLinearOperator(np.linalg.inv(Lmat.toarray()))
Lpinv_factory = lambda w: Lpinv @ operators.MatrixLinearOperator(sps.diags(1.0/w))


## Parameters


In [ ]:
from msolver import VanillaGKS, MMGKS
from pgkbsolver import MMGKB
from fsolver import MMFGK
from f2019solver import MMFGK2019
from fIRWsolver import MMFGKIRW

# Solver imports can load additional BLAS libraries; apply the limit again.
_blas_threads = threadpool_limits(limits=8)
import time

n_its = 150
epsilons = np.logspace(-12, 2, 140)
methods = ["S-GKS", "PS-GKS", "PS-GKB", "FGK", "IRW-FLSQR", "FLSQR-R", "FLSQR-W"]
solver_arguments = dict(noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=None)
common = dict(n_iter=n_its, regparam="dp", x_true=ground_truth,
              ksparsity=None, lambdah_min=1e-7, lambdah_max=1e7)



## Run the complete sweep

Each CSV row records a method and epsilon, its final RRE, Gini index,
regularization parameter, iteration count, and elapsed time.


In [ ]:
def run_epsilon(epsilon, baseline):
    weight_params = {"epsilon": epsilon, "q": 1.0}
    args = dict(common, weight_params=weight_params)
    warm_start = dict(init_dim=5, x0=baseline["x"], lambdah0=baseline["lambdah"],
                      regparam_kwargs={"eta": 1.01})
    gks = MMGKS(A, L, b, **solver_arguments)
    fresh_start = dict(x0=None, lambdah0=None, refresh_basis=False)
    # Flexible methods use the solvers' default DP options.
    calls = [
        ("S-GKS", lambda: gks.solve(**args, **warm_start, min_dim=None, max_dim=None, xker=None)),
        ("PS-GKS", lambda: gks.solve_with_priorconditioning(**args, **warm_start)),
        ("PS-GKB", lambda: MMGKB(A, L, b, **solver_arguments).solve(**args, **fresh_start)),
        ("FGK", lambda: MMFGK(A, L, b, **solver_arguments).solve(**args, **fresh_start)),
        ("IRW-FLSQR", lambda: MMFGKIRW(A, L, Lpinv, b, noise_var=noise_var, K=None).solve(
            **args, **fresh_start, mode="IRW", breakdown_tol=1e-10)),
        ("FLSQR-R", lambda: MMFGK2019(A, L, Lpinv, b, noise_var=noise_var, K=None).solve(
            **args, **fresh_start, mode="R", breakdown_tol=1e-10)),
        ("FLSQR-W", lambda: MMFGK2019(A, L, Lpinv, b, noise_var=noise_var, K=None).solve(
            **args, **fresh_start, mode="W", breakdown_tol=1e-10)),
    ]
    block = []
    for method, solve in calls:
        started = time.perf_counter()
        result = solve()
        block.append({"epsilon": float(epsilon), "method": method,
                      "final_rre": float(result["rre_history"][-1]),
                      "final_gini": float(result["gini_indices"][-1]),
                      "final_mu": float(result["lambda_history"][-1]),
                      "completed_iterations": int(result["n_completed_iterations"]),
                      "elapsed_seconds": time.perf_counter() - started})
    return block

rows = []
with (RESULTS / "figure_05_solver.log").open("w") as log:
    with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
        baseline = VanillaGKS(A, L, b, **solver_arguments).solve(
            **common, init_dim=5, x0=None, lambdah0=1.0,
            weight_params={}, regparam_kwargs={"eta": 1.01}, xker=None)
    for index, epsilon in enumerate(epsilons):
        log.write(f"\nEpsilon {index+1}/{len(epsilons)}: {epsilon:.16g}\n")
        with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            rows.extend(run_epsilon(epsilon, baseline))
        log.flush()
        print(f"Completed epsilon {index+1:3d}/{len(epsilons)}: {epsilon:.4g}", flush=True)

sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(RESULTS / "figure_05.csv", index=False)
sensitivity.head()


## Plot Figure 5

In [ ]:
method_colors = {"S-GKS": "C4", "PS-GKS": "C2", "PS-GKB": "C5",
                 "FGK": "C3", "IRW-FLSQR": "C3", "FLSQR-R": "C3", "FLSQR-W": "C3"}
method_lines = {"S-GKS": "solid", "PS-GKS": "solid", "PS-GKB": "dashed",
                "FGK": "dotted", "IRW-FLSQR": (0, (3, 1, 1, 1, 1, 1)),
                "FLSQR-R": "dashed", "FLSQR-W": "dashdot"}
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
for method in methods:
    frame = sensitivity[sensitivity.method == method].sort_values("epsilon")
    for ax, metric in zip(axs, ["final_rre", "final_gini"]):
        ax.plot(frame.epsilon, frame[metric], label=method,
                color=method_colors[method], ls=method_lines[method], lw=2)
for ax, metric, label in zip(axs, ["RRE", "Gini index"], ["Final RRE", "Final Gini index"]):
    ax.set_xscale("log")
    ax.set_xlabel(r"$\epsilon$")
    ax.set_title(f"Sensitivity of final {metric} ({n_its} iterations)")
    ax.set_ylabel(label)
    ax.legend()
axs[0].set_ylim(0.05, 0.125)
fig.tight_layout()
fig.savefig(RESULTS / "figure_05.png", dpi=250, bbox_inches="tight")
plt.show()
